# L8c: Regularization and Cross-Validation
To prevent overfitting and build models that predict well on new data, we explore __regularization__, which constrains the size of the model parameters, and __cross-validation__, which selects the strength of that constraint by evaluating the model on held-out data.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Estimate linear model parameters with ridge regression:__ Ridge regression adds a penalty on the size of the parameters to the least-squares objective. We'll derive its analytical solution, show how the regularization parameter shrinks the estimated parameters toward zero, and interpret the standard errors for a fixed penalty.
> * __Compute the ridge estimate with the singular value decomposition:__ The singular value decomposition (SVD) writes the ridge estimate as a weighted sum of the right singular vectors of the data matrix. We'll use this form to see which parts of the least-squares estimate the penalty shrinks the most, and why the computation is numerically stable.
> * __Select the regularization parameter with cross-validation:__ Cross-validation estimates how well a model predicts data that were not used to fit it. We'll use k-fold cross-validation to compare candidate values of the regularization parameter and choose the one with the smallest validation error.

We'll demonstrate these ideas with an example, where we estimate a linear model of housing prices with and without regularization. Let's get started!
___

## Examples
We will use the following example to see what regularization does to a model estimated from real data:

> [▶ Let's build a regression model of housing prices with regularization and SVD](CHEME-5800-L8c-Example-RLS-SVD-HousingPriceModel-Fall-2026.ipynb). What does the regularization parameter do to a model estimated from real data? We estimate a linear model of housing prices with ordinary least squares, then with ridge regression, and compare the estimated parameters and the prediction error on training and testing data. We then compute the ridge estimate from the SVD of the training data matrix, find the term that the penalty shrinks the most, and check that the result matches the direct solution.

___

## Concept Review: Overdetermined Linear Regression
Suppose there exists a dataset $\mathcal{D} = \left\{(\mathbf{x}_{i},y_{i}) \mid i = 1,2,\dots,n\right\}$ with $n$ training (labeled) examples, where $\mathbf{x}_{i}\in\mathbb{R}^{m}$ is an $m$-dimensional vector of features (independent input variables) and $y_{i}\in\mathbb{R}$ denotes a scalar response variable (dependent variable).

We can write the linear regression model in matrix-vector form as:
$$
\begin{equation*}
\mathbf{y} = \hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon}
\end{equation*}
$$
where $\hat{\mathbf{X}}$ is an $n\times{p}$ matrix with the augmented features $\hat{\mathbf{x}}_{i}^{\top}$ on the rows (the features plus a `1` for the intercept, so $p=m+1$), the target (output) vector $\mathbf{y}$ is an $n\times{1}$ column vector with entries $y_{i}$, and the error vector $\mathbf{\epsilon}$ is an $n\times{1}$ column vector with entries $\epsilon_{i}$.

Given a data matrix $\hat{\mathbf{X}}\in\mathbb{R}^{n\times{p}}$ that is __overdetermined__, i.e., $n > p$ (more observations than parameters), and an error model $\mathbf{\epsilon}\sim\mathcal{N}(\mathbf{0},\sigma^{2}\;\mathbf{I})$ that follows [a normal distribution](https://en.wikipedia.org/wiki/Normal_distribution) with a mean of zero and variance $\sigma^{2}$, we estimate the model parameters by minimizing the sum of squared errors between the model's estimated outputs and the observed outputs:
$$
\begin{align*}
\hat{\mathbf{\theta}} = \arg\min_{\mathbf{\theta}} \frac{1}{2}\;\lVert~\mathbf{y} - \hat{\mathbf{X}}\;\mathbf{\theta}~\rVert^{2}_{2}
\end{align*}
$$
where $\lVert\star\rVert^{2}_{2}$ is the square of the [L2 vector norm](https://en.wikipedia.org/wiki/Norm_(mathematics)#Euclidean_norm), and $\hat{\mathbf{\theta}}\in\mathbb{R}^{p}$ is the estimated parameter vector. When $\hat{\mathbf{X}}$ has full column rank (i.e., its rank is $r(\hat{\mathbf{X}}) = p$ and $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ is invertible), this problem has the analytical solution:
$$
\begin{align*}
\hat{\mathbf{\theta}} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
\end{align*}
$$

The confidence intervals for the estimated parameters can be computed as follows:

> __Confidence intervals:__ A $(1-\alpha) \times 100\%$ confidence interval for each parameter $\theta_j$ is given by:
>
> $$
> \begin{align*}
> \hat{\theta}_j \pm t_{1-\alpha/2,\nu}\; \text{SE}(\hat{\theta}_j)
> \end{align*}
> $$
>
> where $t_{1-\alpha/2,\nu}$ is the $(1-\alpha/2)$-quantile of a Student $t$ distribution with $\nu = n-p$ degrees of freedom. For large samples, $t_{1-\alpha/2,\nu} \approx 1.96$ for a 95% confidence interval ($\alpha = 0.05$) and $t_{1-\alpha/2,\nu} \approx 3.291$ for a 99.9% confidence interval ($\alpha = 0.001$). The standard error $\text{SE}(\hat{\theta}_j)$ quantifies the uncertainty in the parameter estimate $\hat{\theta}_j$ and is given by:
>
> $$
> \begin{align*}
> \text{SE}(\hat{\theta}_{j}) &= \hat{\sigma}\; \sqrt{\bigl[(\hat{\mathbf X}^\top\hat{\mathbf X})^{-1}\bigr]_{jj}}
> \end{align*}
> $$
>
> where $\hat{\sigma}^{2} = \lVert~\mathbf{r}~\rVert^{2}_{2}/(n-p)$ is the error variance estimated from the residuals $\mathbf{r} = \mathbf{y} - \hat{\mathbf{X}}\hat{\mathbf{\theta}}$.

We developed these results in the L7c lecture and used them to build a linear model of housing prices in the L7d lab. That model, estimated without regularization, is our starting point today.

___

## Regularized linear regression
In the __overdetermined case__, we can add a regularization term to the objective function to prevent overfitting and improve generalization. This approach is called __regularized linear regression__.

> __What is overfitting?__ Overfitting occurs when a model learns the noise in the training data instead of the underlying pattern. This leads to poor performance on unseen data because the model fails to generalize beyond the training set.

There are several types of regularization techniques, but we will focus on __ridge regression__ (also known as Tikhonov regularization or L2 regularization). The ridge regression problem is given by:
$$
\begin{align*}
\hat{\mathbf{\theta}}_{\delta} = \arg\min_{\mathbf{\theta}}\left( \frac{1}{2}\;\lVert~\mathbf{y} - \hat{\mathbf{X}}\;\mathbf{\theta}~\rVert^{2}_{2} + \frac{\delta}{2}\;\lVert~\mathbf{\theta}~\rVert^{2}_{2}\right)
\end{align*}
$$
where $\delta\geq 0$ is the __regularization parameter__, which controls the strength of the penalty. The first term measures the sum of squared errors, while the second term penalizes large parameter values. Setting the gradient of this objective with respect to $\mathbf{\theta}$ to zero gives the analytical solution for the optimal parameters:
$$
\begin{align*}
\hat{\mathbf{\theta}}_{\delta} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
\end{align*}
$$
When $\delta > 0$, the matrix $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}$ is invertible for any data matrix, so the ridge estimate exists even when $\hat{\mathbf{X}}$ does not have full column rank.

This solution can also be expressed in terms of the true parameters $\mathbf{\theta}$ and the error model $\mathbf{\epsilon}$:
$$
\begin{align*}
\hat{\mathbf{\theta}}_{\delta} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}(\hat{\mathbf{X}}\;\mathbf{\theta} + \mathbf{\epsilon}) \\
\hat{\mathbf{\theta}}_{\delta} &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\;\mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon} \\
\hat{\mathbf{\theta}}_{\delta} &= \underbrace{\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}}_{\text{Shrinkage}\;\mathbf{P}}\;\mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon} \\
\hat{\mathbf{\theta}}_{\delta} &= \mathbf{P}\;\mathbf{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{\epsilon}\quad\blacksquare\\
\end{align*}
$$

> __Key insight:__ The penalty shrinks the estimated parameters toward zero. When $\delta = 0$, the shrinkage matrix is $\mathbf{P} = \mathbf{I}$ and we recover the unbiased OLS estimate. When $\delta > 0$, the expected estimate is $\mathbb{E}[\hat{\mathbf{\theta}}_{\delta}] = \mathbf{P}\;\mathbf{\theta}$, which has a smaller norm than $\mathbf{\theta}$, so the ridge estimate is biased. In exchange, the estimated parameters vary less from one dataset to the next, which helps prevent overfitting by discouraging complex models that fit the training data too closely. This exchange is called the __bias-variance tradeoff__.

__In practice:__ The penalty treats every parameter the same way, so its effect depends on the units of the features. We typically put the features on a common scale before fitting a ridge model, and often leave the intercept out of the penalty. We'll do both in the L8d lab.

### Sampling variability of the ridge estimate
How does the ridge penalty change the standard errors? Suppose we repeat the observations with the same data matrix, the same error variance, and a fixed regularization parameter. Only the error term in the ridge decomposition is random. Using $\text{Var}(\mathbf{A}\mathbf{z}) = \mathbf{A}\;\text{Var}(\mathbf{z})\;\mathbf{A}^{\top}$ and $\text{Var}(\mathbf{\epsilon}) = \sigma^{2}\;\mathbf{I}$, as we did in the L7c lecture, the covariance matrix of the ridge estimate is given by:

$$
\begin{align*}
\text{Var}(\hat{\mathbf{\theta}}_{\delta}) &= \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\;(\sigma^{2}\;\mathbf{I})\;\hat{\mathbf{X}}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\\
&= \sigma^{2}\; \underbrace{\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I}\right)^{-1}}_{ =\;\mathbf{A}_{\delta}}.\quad\blacksquare
\end{align*}
$$

When the data matrix has full column rank, we can estimate the unknown error variance with the OLS residual estimate $\hat{\sigma}^{2}$ defined in the concept review. This gives the estimated standard errors:

$$
\begin{align*}
\text{SE}(\hat{\theta}_{\delta,j}) &= \hat{\sigma}\; \sqrt{\bigl[\mathbf{A}_{\delta}\bigr]_{jj}}.
\end{align*}
$$

The standard error measures the spread of a fitted coefficient around its expected value, $[\mathbf{P}\;\mathbf{\theta}]_{j}$, due to the observation errors. When $\delta = 0$, the matrix $\mathbf{A}_{\delta}$ reduces to $(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}$ and we recover the OLS standard errors.

With regularization, a confidence interval for the original coefficient $\theta_{j}$ must also account for shrinkage bias, whose size depends on the penalty. Choosing the penalty by cross-validation adds uncertainty from that choice. The calculation above describes the variability for a fixed penalty; see [Jonathan Taylor's ridge regression notes](https://web.stanford.edu/class/stats305b/notes/Ridge.html#why-do-we-regularize) for the sampling distribution and the bias-variance decomposition.

___

## SVD solution for regularized least squares
Let the singular value decomposition (SVD) of the $n\times{p}$ data matrix $\hat{\mathbf{X}}$ be given by:
$$
\begin{equation*}
\hat{\mathbf{X}} = \mathbf{U}\;\mathbf{S}\;\mathbf{V}^{\top}
\end{equation*}
$$
where $\mathbf{U} \in \mathbb{R}^{n \times n}$ is an orthogonal matrix, $\mathbf{S} \in \mathbb{R}^{n \times p}$ is a rectangular diagonal matrix with the singular values $\sigma_{1}\geq\cdots\geq\sigma_{r}>0$ on its first $r=r(\hat{\mathbf{X}})$ diagonal entries and zeros elsewhere, and $\mathbf{V} \in \mathbb{R}^{p \times p}$ is an orthogonal matrix. This is the SVD notation from the L7a and L7c lectures. Substituting the SVD into the ridge solution, the regularized least-squares estimate (ridge regression) of the unknown parameter vector $\mathbf{\theta}$ is given by:
$$
\begin{equation*}
\hat{\mathbf{\theta}}_{\delta} = \mathbf{V}\left(\mathbf{S}^{\top}\mathbf{S}+\delta\;\mathbf{I}\right)^{-1}\mathbf{S}^{\top}\mathbf{U}^{\top}\mathbf{y}
\end{equation*}
$$
or equivalently, in index notation:
$$
\boxed{
\begin{equation*}
\hat{\mathbf{\theta}}_{\delta} = \sum_{i=1}^{r(\hat{\mathbf{X}})}\left(\frac{\sigma_{i}}{\sigma_{i}^{2}+\delta}\right)\left(\mathbf{u}_{i}^{\top}\mathbf{y}\right)\mathbf{v}_{i}\quad\blacksquare
\end{equation*}}
$$
where $r(\hat{\mathbf{X}})\leq\min(n,p)$ is the rank of the data matrix $\hat{\mathbf{X}}$, $\mathbf{u}_{i}$ and $\mathbf{v}_{i}$ are the $i$-th columns of $\mathbf{U}$ and $\mathbf{V}$, respectively, $\sigma_{i}>0$ is the $i$-th singular value, and $\delta \geq 0$ is the regularization parameter.

How does this compare with the unregularized estimate? In the L7c lecture, each term of the SVD solution was weighted by $1/\sigma_{i}$. The ridge weight is the same $1/\sigma_{i}$ multiplied by a __filter factor__ $f_{i}$ between zero and one:
$$
\begin{equation*}
\frac{\sigma_{i}}{\sigma_{i}^{2}+\delta} = \underbrace{\left(\frac{\sigma_{i}^{2}}{\sigma_{i}^{2}+\delta}\right)}_{\text{filter factor}\;f_{i}}\frac{1}{\sigma_{i}}
\end{equation*}
$$

> __Key insights:__
>
> * __Shrinkage effect:__ The regularization parameter $\delta$ shrinks the contribution of each term in the sum by its filter factor. Terms with $\sigma_{i}^{2}\gg\delta$ have $f_{i}\approx{1}$ and are nearly unchanged, while terms with $\sigma_{i}^{2}\ll\delta$ have $f_{i}\approx{0}$. The smaller singular values, which amplify the noise in the unregularized estimate, are shrunk more aggressively.
> * __Numerical stability:__ This SVD formulation is more numerically stable than directly computing $(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\;\mathbf{I})^{-1}$, especially when $\hat{\mathbf{X}}$ is ill-conditioned, i.e., has a large condition number. The [condition number notebook](CHEME-5800-L8c-Theory-ConditionNumber-Fall-2026.ipynb) defines this term and explains why it matters.
> * __Limiting cases:__ When $\delta = 0$, we recover the unregularized solution, and as $\delta \to \infty$, all the parameters shrink to zero.

Let's revisit our housing example, but in the context of regularization and SVD.

> __Example__
>
> [▶ Let's build a regression model of housing prices with regularization and SVD](CHEME-5800-L8c-Example-RLS-SVD-HousingPriceModel-Fall-2026.ipynb). What does the regularization parameter do to a model estimated from real data? We estimate a linear model of housing prices with ordinary least squares, then with ridge regression, and compare the estimated parameters and the prediction error on training and testing data. We then compute the ridge estimate from the SVD of the training data matrix, find the term that the penalty shrinks the most, and check that the result matches the direct solution.

___

## Cross-validation
Now, the question is: how do we select the regularization parameter $\delta$? The training error is always smallest at $\delta = 0$, so it cannot make the choice for us. The answer is that we can use cross-validation techniques to tune the hyperparameter $\delta$ by evaluating the model's performance on held-out validation data.

__Cross-validation (CV)__ is a statistical method used to estimate the performance of machine learning models. It is primarily used in scenarios where the goal is to assess how the results of a statistical analysis will generalize to an independent dataset.

> __k-fold cross-validation:__ The most common form of cross-validation is k-fold cross-validation, where the dataset is divided into $k$ subsets (or "folds"). The model is trained on $k-1$ folds and validated on the remaining fold. This process is repeated $k$ times, with each fold serving as the validation set once.

The performance metrics from each iteration are then averaged to provide a more robust estimate of the model's generalization performance. To select $\delta$, we repeat this procedure for each candidate value and choose the one with the smallest average validation error.

__In practice:__ Anything we compute from the data before fitting, such as the feature means and standard deviations used for scaling, is computed from the training folds only. This keeps the validation fold unseen until we evaluate the model.

___

## Lab
In [L8d](../L8d/CHEME-5800-L8d-Lab-CrossValidationAndResidualModelChecking-Fall-2026.ipynb), we will implement k-fold cross-validation, use it to select the regularization parameter of a ridge regression model of housing prices, and then check the residuals of the selected model.

___

## Summary
We added a penalty on the size of the parameters to the least-squares problem, wrote the ridge estimate in terms of the singular value decomposition, and used cross-validation to select the regularization parameter.

> __Key Takeaways:__
>
> * __Ridge regression:__ We added a penalty on the squared norm of the parameters to the least-squares objective and derived the analytical solution for the ridge estimate. The regularization parameter shrinks the estimated parameters toward zero, which introduces bias but makes the estimates less sensitive to the noise in the training data.
>
> * __Ridge regression with the SVD:__ We wrote the ridge estimate as a weighted sum of the right singular vectors, where each term of the least-squares estimate is multiplied by a filter factor between zero and one. The terms with small singular values, which amplify noise, are shrunk the most, and the computation remains stable when the data matrix is ill-conditioned.
>
> * __Cross-validation:__ We used k-fold cross-validation to estimate how well a model predicts data that were not used to fit it, training on all but one fold and validating on the fold left out. Averaging the validation error across the folds lets us compare candidate values of the regularization parameter and select the one that generalizes best.

These techniques form the foundation for practical machine learning applications where preventing overfitting and selecting appropriate model complexity are critical for reliable predictions.

___